# Course 12 Lab — Enterprise AI Operating Model and Leadership

**Capstone:** turn Northstar's 70 GenAI proofs of concept into a small, governed,
evidence-backed production portfolio and a defensible 12-month operating-model roadmap.

## 1. Scope, safety, and authority

This notebook is deterministic, credential-free, and synthetic. It teaches decision mechanics;
it is not a real investment recommendation or compliance assessment. Scores and models propose.
Trusted evidence processes and accountable authorities validate, allocate, accept risk,
authorize production, and verify outcomes.

In [1]:
from dataclasses import replace

from lab import (
    DelegationContract,
    DissentRecord,
    DissentStatus,
    ExceptionRequest,
    MaturityLevel,
    PortfolioDisposition,
    RiskTier,
    assess_candidate,
    assess_maturity,
    assess_roadmap,
    control_profile,
    demo_candidates,
    demo_decision_rights,
    demo_evidence,
    demo_investments,
    demo_maturity_evidence,
    demo_roadmap,
    demo_services,
    evaluate_investment,
    issue_exception,
    portfolio_metrics,
    run_demo_operating_model,
    select_portfolio,
    validate_decision_rights,
    validate_delegation,
    validate_dissent,
    validate_exception,
    validate_service_catalog,
)

TRUSTED = frozenset({"portfolio-evidence-office"})
NOW = 1_300
print("Course 12 operating-model lab ready")

Course 12 operating-model lab ready


## 2. The operating model is a decision system

In [2]:
loops = [
    "strategy and risk appetite", "intake and inventory", "portfolio allocation",
    "product and platform delivery", "assurance and authorization",
    "outcome, economics and learning feedback",
]
for index, item in enumerate(loops, start=1):
    print(index, item)

1 strategy and risk appetite
2 intake and inventory
3 portfolio allocation
4 product and platform delivery
5 assurance and authorization
6 outcome, economics and learning feedback


Team boxes are insufficient. The system needs records, authority, service interfaces,
constraints, feedback, and a path to stop or change work.

## 3. Load the complete 70-PoC portfolio

In [3]:
candidates = demo_candidates()
evidence = demo_evidence(candidates)
print("candidates:", len(candidates), "evidence records:", len(evidence))
assert len(candidates) == 70 and len(evidence) == 210

candidates: 70 evidence records: 210


## 4. Inspect the intake contract

In [4]:
for item in candidates[:5]:
    print(
        item.candidate_id, item.domain, item.risk_tier,
        f"value=${item.expected_annual_value:,.0f}",
        f"cost=${item.annual_cost:,.0f}", "owner=", item.owner,
    )

poc-01 underwriting low value=$165,000 cost=$105,000 owner= owner-1
poc-02 claims low value=$210,000 cost=$130,000 owner= owner-2
poc-03 service low value=$255,000 cost=$155,000 owner= owner-3
poc-04 finance moderate value=$300,000 cost=$180,000 owner= owner-4
poc-05 operations low value=$345,000 cost=$205,000 owner= owner-5


Every record has an exact version, owner, outcome, risk, data state, resources and dependencies.
Evidence is stored separately so a claim cannot validate itself.

## 5. Establish a naive baseline

In [5]:
naive = sorted(candidates, key=lambda item: item.sunk_cost, reverse=True)[:8]
print("naive sunk-cost continuation:", [item.candidate_id for item in naive])
print("failure: prior spend is not prospective value, evidence, safety, capacity or authority")

naive sunk-cost continuation: ['poc-04', 'poc-09', 'poc-14', 'poc-19', 'poc-24', 'poc-29', 'poc-34', 'poc-39']
failure: prior spend is not prospective value, evidence, safety, capacity or authority


## 6. Inspect hard gates before scoring

In [6]:
prohibited = next(item for item in candidates if item.risk_tier is RiskTier.PROHIBITED)
prohibited_result = assess_candidate(
    prohibited, evidence, trusted_producers=TRUSTED, now=NOW
)
print(prohibited.candidate_id, prohibited_result)
assert prohibited_result.disposition is PortfolioDisposition.STOP

poc-23 CandidateAssessment(candidate_id='poc-23', disposition=<PortfolioDisposition.STOP: 'stop'>, score=None, reasons=('prohibited-use',))


A high expected value cannot compensate for prohibited use, missing ownership, or unapproved
sensitive data. No weighted score is calculated after a hard failure.

## 7. Assess every candidate

In [7]:
assessments = tuple(
    assess_candidate(item, evidence, trusted_producers=TRUSTED, now=NOW)
    for item in candidates
)
counts = {
    state.value: sum(item.disposition is state for item in assessments)
    for state in PortfolioDisposition
}
print(counts)

{'fund': 54, 'experiment': 10, 'hold': 0, 'stop': 6, 'refer': 0}


## 8. Inspect the score evidence chain

In [8]:
first = candidates[0]
first_proof = [item for item in evidence if item.candidate_id == first.candidate_id]
print(first)
for item in first_proof:
    print(item.metric, item.value, item.unit, item.producer, item.expires_at)
print(next(item for item in assessments if item.candidate_id == first.candidate_id))

PortfolioCandidate(candidate_id='poc-01', version='v1', domain='underwriting', sponsor='sponsor-1', owner='owner-1', outcome='reduce cycle time for workflow 1', strategic_alignment=0.54, expected_annual_value=165000.0, feasibility=0.55, reuse_potential=0.49, risk_tier=<RiskTier.LOW: 'low'>, data_approved=True, annual_cost=105000.0, team_capacity=2, duplicate_group=None, dependencies=(), sunk_cost=50000.0)
annual-value 165000.0 usd/year portfolio-evidence-office 2000
feasibility 0.55 score portfolio-evidence-office 2000
outcome-confidence 0.5599999999999999 score portfolio-evidence-office 2000
CandidateAssessment(candidate_id='poc-01', disposition=<PortfolioDisposition.FUND: 'fund'>, score=0.4934, reasons=('eligible-for-portfolio',))


## 9. Failure injection: missing evidence

In [9]:
missing = tuple(
    item for item in evidence
    if not (item.candidate_id == first.candidate_id and item.metric == "feasibility")
)
missing_result = assess_candidate(first, missing, trusted_producers=TRUSTED, now=NOW)
print(missing_result)
assert missing_result.disposition is PortfolioDisposition.REFER

CandidateAssessment(candidate_id='poc-01', disposition=<PortfolioDisposition.REFER: 'refer'>, score=None, reasons=('missing-evidence:feasibility',))


Missing evidence is not zero and not a pass. `refer` sends the question to an evidence owner or
a bounded experiment without pretending the candidate failed its outcome.

## 10. Failure injection: stale and untrusted evidence

In [10]:
changed = list(evidence)
index = next(
    i for i, item in enumerate(changed)
    if item.candidate_id == first.candidate_id and item.metric == "annual-value"
)
changed[index] = replace(changed[index], producer="sponsor-slide", expires_at=1_200)
untrusted = assess_candidate(first, changed, trusted_producers=TRUSTED, now=NOW)
print(untrusted.reasons)
assert untrusted.disposition is PortfolioDisposition.REFER

('untrusted-evidence:annual-value',)


## 11. Create a bounded experiment

In [11]:
candidate = replace(first, feasibility=0.50)
experiment_proof = demo_evidence((candidate,))
experiment = assess_candidate(candidate, experiment_proof, trusted_producers=TRUSTED, now=NOW)
print(experiment)
assert experiment.disposition is PortfolioDisposition.EXPERIMENT

CandidateAssessment(candidate_id='poc-01', disposition=<PortfolioDisposition.EXPERIMENT: 'experiment'>, score=0.4834, reasons=('bounded-learning-needed',))


An experiment buys information under a smaller spend and capacity envelope. It does not receive
production authorization and creates no obligation to continue.

## 12. Select under enterprise constraints

In [12]:
selection = select_portfolio(
    candidates, assessments, budget=2_500_000, capacity=45, max_high_risk=2
)
print("funded:", selection.funded)
print("experiments:", selection.experiments)
print("held:", len(selection.held), "stopped:", len(selection.stopped))
print("spend:", selection.annual_cost, "capacity:", selection.team_capacity)
assert selection.annual_cost <= 2_500_000 and selection.team_capacity <= 45

funded: ('poc-34', 'poc-48', 'poc-59', 'poc-09', 'poc-17', 'poc-19', 'poc-33', 'poc-41', 'poc-47', 'poc-16', 'poc-04', 'poc-64', 'poc-53', 'poc-08', 'poc-26', 'poc-05', 'poc-13', 'poc-32')
experiments: ('poc-49', 'poc-56', 'poc-28', 'poc-35')
held: 42 stopped: 6
spend: 2492000.0 capacity: 42


## 13. Inspect dependency and duplication reasons

In [13]:
interesting = {
    key: value for key, value in selection.reasons.items()
    if value in {"dependency-not-funded", "duplicate-capability", "high-risk-cap-reached"}
}
print(interesting)

{'poc-27': 'duplicate-capability', 'poc-39': 'duplicate-capability', 'poc-18': 'duplicate-capability', 'poc-24': 'duplicate-capability', 'poc-57': 'duplicate-capability', 'poc-03': 'duplicate-capability', 'poc-54': 'duplicate-capability', 'poc-06': 'duplicate-capability', 'poc-20': 'dependency-not-funded', 'poc-12': 'duplicate-capability', 'poc-66': 'duplicate-capability', 'poc-45': 'duplicate-capability', 'poc-10': 'dependency-not-funded', 'poc-51': 'duplicate-capability', 'poc-15': 'duplicate-capability', 'poc-60': 'duplicate-capability', 'poc-36': 'duplicate-capability', 'poc-30': 'dependency-not-funded'}


Portfolio value is not the sum of isolated rankings. Shared capabilities, dependencies,
concentration, scarce teams, and learning options alter the best set.

## 14. Prove sunk cost does not affect selection

In [14]:
changed_candidates = tuple(
    replace(item, sunk_cost=item.sunk_cost + 5_000_000) for item in candidates
)
changed_selection = select_portfolio(
    changed_candidates, assessments, budget=2_500_000, capacity=45, max_high_risk=2
)
print("same funded set:", changed_selection.funded == selection.funded)
assert changed_selection.funded == selection.funded

same funded set: True


## 15. Map risk to a production path

In [15]:
for tier in RiskTier:
    profile = control_profile(tier)
    print(tier, profile.mode, sorted(profile.required_controls), profile.independent_review)

low self-service ['evaluation', 'inventory', 'owner'] False
moderate guarded ['evaluation', 'inventory', 'monitoring', 'owner', 'threat-model'] False
high independent-review ['evaluation', 'human-oversight', 'inventory', 'monitoring', 'owner', 'threat-model'] True
prohibited prohibited [] True


Risk changes the control path. It never removes the need for ownership or outcome evidence, and
a prohibited path cannot become self-service through an exception.

## 16. Issue a narrow exception

In [16]:
request = ExceptionRequest(
    "request-1", "service-owner", first.candidate_id, first.version,
    "telemetry-policy", "regional-dashboard", "migration window",
    ("daily manual review", "restricted access"),
)
receipt = issue_exception(
    request, approver="control-owner",
    authorized_approvers=frozenset({"control-owner"}),
    policy_version="policy-v7", now=NOW, ttl=90,
)
print(receipt)
validate_exception(request, receipt, current_policy_version="policy-v7", now=1_350)

ExceptionReceipt(receipt_id='exception-aec52d8c577e', request_digest='aec52d8c577e7f3143a57018d1db6289a38657639f044c5a21f3df05741b52a8', approver='control-owner', policy_version='policy-v7', issued_at=1300, expires_at=1390, used=False)


## 17. Failure injection: altered exception subject

In [17]:
altered = replace(request, candidate_version="v2")
try:
    validate_exception(altered, receipt, current_policy_version="policy-v7", now=1_350)
except ValueError as error:
    print("rejected:", error)
else:
    raise AssertionError("altered exception was accepted")

rejected: exception subject changed


## 18. Validate decision rights

In [18]:
rights = demo_decision_rights()
validate_decision_rights(rights)
for item in rights:
    print(
        item.kind, "A=", item.accountable, "R=", item.responsible,
        "escalate=", item.escalation_owner,
    )

product-outcome A= product R= ('domain-team',) escalate= portfolio-council
platform-standard A= platform R= ('platform-team',) escalate= cto
risk-acceptance A= risk R= ('risk-office',) escalate= cro
production-release A= service-owner R= ('delivery-team',) escalate= engineering-vp
provider-contract A= procurement R= ('vendor-office',) escalate= cfo
policy-exception A= control-owner R= ('governance',) escalate= cro


The matrix covers product outcome, platform standard, risk acceptance, production release,
provider contract and policy exception exactly once. Organizational labels document agreement;
runtime identity and policy still enforce technical authorization.

## 19. Failure injection: accountability becomes informed-only

In [19]:
broken_rights = list(rights)
broken_rights[0] = replace(broken_rights[0], informed=(broken_rights[0].accountable,))
try:
    validate_decision_rights(broken_rights)
except ValueError as error:
    print("rejected:", error)

rejected: accountable owner cannot be informed-only:product-outcome


## 20. Validate the AI service catalogue

In [20]:
services = demo_services()
validate_service_catalog(services)
for service in services:
    print(service.service_id, service.outcome, service.service_level, service.exit_path)

model-gateway governed inference 99.9% monthly direct provider adapter
evaluation-service release evidence one business day portable evaluation bundle
risk-review independent decision five business days external assurance path


## 21. Failure injection: paved road without exit

In [21]:
broken_service = replace(services[0], exit_path="")
try:
    validate_service_catalog((broken_service,))
except ValueError as error:
    print("rejected:", error)

rejected: incomplete service contract:model-gateway


A portal entry is not a service. Consumers need eligibility, onboarding, SLO, support, quota,
lifecycle, unit cost, deprecation and a governed escape path.

## 22. Assess nine capability dimensions

In [22]:
maturity = assess_maturity(
    demo_maturity_evidence(),
    trusted_producers=frozenset({"assurance-office"}), now=NOW,
)
for dimension, level in maturity.levels.items():
    print(f"{dimension:12} {level}")
print("overall floor:", maturity.overall)
assert maturity.overall is MaturityLevel.PROVISIONAL

strategy     scalable
product      operational
data         operational
engineering  scalable
evaluation   operational
security     scalable
operations   operational
platform     operational
people       provisional
overall floor: provisional


## 23. Failure injection: missing maturity proof

In [23]:
incomplete_maturity = assess_maturity(
    demo_maturity_evidence()[:-1],
    trusted_producers=frozenset({"assurance-office"}), now=NOW,
)
print(incomplete_maturity.overall, incomplete_maturity.gaps)
assert incomplete_maturity.overall is None

None ('missing-or-duplicate:people',)


Maturity is an evidenced capability claim, not a workshop mood. The floor exposes a weak people
system that an average would conceal.

## 24. Validate the 12-month roadmap

In [24]:
roadmap = demo_roadmap()
roadmap_result = assess_roadmap(
    roadmap,
    quarterly_budget={1: 200_000, 2: 250_000, 3: 350_000, 4: 250_000},
    quarterly_capacity={1: 3, 2: 4, 3: 6, 4: 4},
)
print("valid:", roadmap_result.valid)
print("quarterly cost:", roadmap_result.quarterly_cost)
assert roadmap_result.valid

valid: True
quarterly cost: {1: 120000.0, 2: 180000.0, 3: 300000.0, 4: 220000.0}


## 25. Trace roadmap outcomes and kill criteria

In [25]:
for item in roadmap:
    print(
        f"Q{item.quarter}", item.initiative_id, item.outcome_metric,
        f"{item.baseline} -> {item.target}", "kill:", item.kill_criteria,
    )

Q1 decision-rights decisions with one accountable owner 0.4 -> 1.0 kill: ownership conflicts remain
Q2 portfolio-gate PoCs with outcome and evidence 0.2 -> 0.9 kill: false-stop rate exceeds 10%
Q3 paved-road median compliant path lead days 45 -> 10 kill: adoption below 30%
Q4 scale-review scaled compliant outcomes 0 -> 6 kill: unit value remains negative


## 26. Failure injection: dependency in the same quarter

In [26]:
broken_roadmap = list(roadmap)
broken_roadmap[1] = replace(broken_roadmap[1], quarter=1)
broken_result = assess_roadmap(
    broken_roadmap,
    quarterly_budget={1: 1_000_000, 2: 1_000_000, 3: 1_000_000, 4: 1_000_000},
    quarterly_capacity={1: 20, 2: 20, 3: 20, 4: 20},
)
print(broken_result.reasons)
assert not broken_result.valid

('invalid-dependency:portfolio-gate:decision-rights',)


## 27. Compare low, base, and high investment cases

In [27]:
investment_results = tuple(
    evaluate_investment(item, discount_rate=0.08) for item in demo_investments()
)
for item in investment_results:
    print(item)
assert investment_results[0].npv < investment_results[1].npv < investment_results[2].npv

InvestmentResult(name='low', npv=-1029663.92, discounted_benefit=404378.14, discounted_cost=1434042.07, benefit_cost_ratio=0.282, break_even_year=None)
InvestmentResult(name='base', npv=-142953.31, discounted_benefit=1373647.31, discounted_cost=1516600.62, benefit_cost_ratio=0.9057, break_even_year=None)
InvestmentResult(name='high', npv=1040491.79, discounted_benefit=2639650.97, discounted_cost=1599159.17, benefit_cost_ratio=1.6506, break_even_year=3)


Benefits are discounted by adoption as well as time. The scenarios expose sensitivity; they do
not promise that modelled productivity becomes cash or that correlation is causal impact.

## 28. Preserve evidence-backed dissent

In [28]:
dissent = DissentRecord(
    "portfolio-2027", "proposal-digest", "security-lead", ("threat-model-12",),
    "portfolio-chair", "stage with additional monitoring",
    DissentStatus.ACCEPTED_RISK, ("supplier concentration",),
)
validate_dissent(dissent)
print(dissent)

DissentRecord(decision_id='portfolio-2027', proposal_digest='proposal-digest', objector='security-lead', evidence_ids=('threat-model-12',), decision_owner='portfolio-chair', response='stage with additional monitoring', status=<DissentStatus.ACCEPTED_RISK: 'accepted-risk'>, unresolved_risks=('supplier concentration',), escalation_owner=None)


## 29. Failure injection: resolved status hides risk

In [29]:
try:
    validate_dissent(replace(dissent, status=DissentStatus.RESOLVED))
except ValueError as error:
    print("rejected:", error)

rejected: resolved dissent cannot hide unresolved risks


## 30. Define bounded delegation and mentoring

In [30]:
delegation = DelegationContract(
    "draft and defend the evaluation service standard", "staff-engineer",
    "may propose and revise; control owner approves",
    ("principal architect", "policy repository", "consumer interviews"),
    7, "hard constraint or unresolved owner conflict",
    "review-ready standard, conformance test and migration note",
    "lead a cross-functional evidence review",
)
validate_delegation(delegation)
print(delegation)

DelegationContract(objective='draft and defend the evaluation service standard', delegate='staff-engineer', decision_boundary='may propose and revise; control owner approves', resources=('principal architect', 'policy repository', 'consumer interviews'), check_in_days=7, escalation_trigger='hard constraint or unresolved owner conflict', definition_of_done='review-ready standard, conformance test and migration note', learning_outcome='lead a cross-functional evidence review')


Delegation names the objective, authority boundary, resources, check-ins, escalation, completion
and learning. It grows decision capacity without silently transferring production authority.

## 31. Evaluate portfolio metrics with explicit denominators

In [31]:
metrics = portfolio_metrics(candidates, selection)
print(metrics)
assert metrics.total == 70
assert metrics.funded_rate == round(len(selection.funded) / 70, 4)

PortfolioMetrics(total=70, funded_rate=0.2571, stopped_early_rate=0.0857, evidence_gap_rate=0.0, shared_capability_rate=0.6111, spend_per_funded_outcome=138444.44)


## 32. Compare baseline and governed system

In [32]:
comparison = {
    "naive": {"hard_gates": False, "evidence_state": False, "constraints": 1, "audit": False},
    "governed": {"hard_gates": True, "evidence_state": True, "constraints": 5, "audit": True},
}
for name, attributes in comparison.items():
    print(name, attributes)

naive {'hard_gates': False, 'evidence_state': False, 'constraints': 1, 'audit': False}
governed {'hard_gates': True, 'evidence_state': True, 'constraints': 5, 'audit': True}


## 33. Run the complete operating-model demonstration

In [33]:
result = run_demo_operating_model()
print("candidate count:", result["candidate_count"])
print("maturity floor:", result["maturity"].overall)
print("roadmap valid:", result["roadmap"].valid)
print("investment cases:", [item.name for item in result["investments"]])

candidate count: 70
maturity floor: provisional
roadmap valid: True
investment cases: ['low', 'base', 'high']


## 34. Production authorization boundary

In [34]:
boundary = {
    "model_or_score": "propose, cluster, summarize, challenge",
    "evidence_process": "validate identity, version, unit, freshness and independence",
    "accountable_authority": "allocate, accept risk, contract, authorize, sunset",
    "control_system": "enforce, execute, observe, reconcile and record",
}
for actor, responsibility in boundary.items():
    print(actor, "->", responsibility)

model_or_score -> propose, cluster, summarize, challenge
evidence_process -> validate identity, version, unit, freshness and independence
accountable_authority -> allocate, accept risk, contract, authorize, sunset
control_system -> enforce, execute, observe, reconcile and record


No notebook output is an approval. A production implementation needs authenticated identities,
current policy, durable state, atomic approvals, protected evidence, retries, audit and appeal.

## 35. Production upgrade checklist

In [35]:
upgrades = [
    "versioned authorized portfolio registry", "signed evidence provenance",
    "policy enforcement and atomic exception use", "finance and capacity integration",
    "live service SLO and support telemetry", "quarterly outcome and sunset review",
    "least-privilege access, retention and audit", "metric definitions and anti-gaming review",
]
for item in upgrades:
    print("[ ]", item)

[ ] versioned authorized portfolio registry
[ ] signed evidence provenance
[ ] policy enforcement and atomic exception use
[ ] finance and capacity integration
[ ] live service SLO and support telemetry
[ ] quarterly outcome and sunset review
[ ] least-privilege access, retention and audit
[ ] metric definitions and anti-gaming review


## 36. Executive narrative exercise

In [36]:
narrative = {
    "problem": "70 PoCs exceed ownership, evidence and delivery capacity",
    "decision": (
        f"fund {len(selection.funded)}, experiment {len(selection.experiments)}, "
        f"stop {len(selection.stopped)}"
    ),
    "investment": "approve staged low/base/high envelope, not one guaranteed ROI",
    "risk": "hard gates, tiered controls and independently owned exceptions",
    "milestones": "decision rights -> portfolio gate -> paved road -> scale review",
    "ask": "fund evidence and shared capability before broad production expansion",
}
for key, value in narrative.items():
    print(f"{key:12}: {value}")

problem     : 70 PoCs exceed ownership, evidence and delivery capacity
decision    : fund 18, experiment 4, stop 6
investment  : approve staged low/base/high envelope, not one guaranteed ROI
risk        : hard gates, tiered controls and independently owned exceptions
milestones  : decision rights -> portfolio gate -> paved road -> scale review
ask         : fund evidence and shared capability before broad production expansion


## 37. Exercises

1. Add a domain-concentration constraint while preserving deterministic reason codes.
2. Design a bounded experiment for one referred high-value candidate.
3. Add opportunity cost and forecast error to the investment scenarios.
4. Write a non-waivable control and an identity-backed enforcement design.
5. Define a platform experience metric that cannot be satisfied by forced adoption.
6. Rehearse a five-minute oral defense and answer an evidence-backed dissent.

## 38. Summary

You converted a demo backlog into an operating system for enterprise AI decisions. The system
gates non-compensatory risk, qualifies evidence, separates learning from scaling, selects under
real constraints, defines service and authority boundaries, preserves dissent, funds a
dependency-aware roadmap, and keeps uncertainty visible. That is Staff-level leverage.